<a class="anchor" id='import'>
<font color = '#006400'>
    
# **1. Data Integration** </font>
</a>

<a class="anchor" id='lib'></a>
<font color = '#008000'>

## **1.1. Import the needed libraries** </font>

In [ ]:
import pandas as pd

<a class="anchor" id='lib'></a>
<font color = '#008000'>

## **1.2. Integrate the datasets into the notebook** </font>

In [ ]:
#Import u.data dataset
ratings_path = "/Users/cristiana/Downloads/ml-100k/u.data"
columns = ['userId', 'movieId', 'rating', 'timestamp']

ratings = pd.read_csv(ratings_path, sep='\t', names=columns)

In [ ]:
#Import u.item dataset
items_path = "/Users/cristiana/Downloads/ml-100k/u.item"

item_columns = [
    'movieId', 'title', 'release_date', 'video_release_date', 'IMDb_URL',
    'unknown', 'Action', 'Adventure', 'Animation', "Children's", 'Comedy',
    'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror',
    'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western'
]

movies = pd.read_csv(items_path, sep='|', names=item_columns, encoding='latin-1')


In [ ]:
#Import u.user dataset
users_path = "/Users/cristiana/Downloads/ml-100k/u.user"

user_columns = ['userId', 'age', 'gender', 'occupation', 'zip_code']

users = pd.read_csv(users_path, sep='|', names=user_columns)

In [ ]:
#Import u.genre dataset
genre_path = "/Users/cristiana/Downloads/ml-100k/u.genre"

genre_columns = ['genre', 'genreId']

genres = pd.read_csv(genre_path, sep='|', names=genre_columns, encoding='latin-1')


In [ ]:
#Import u.occupation dataset
occupation_path = "/Users/cristiana/Downloads/ml-100k/u.occupation"

occupations = pd.read_csv(occupation_path, names=['occupation'], encoding='latin-1')

<a class="anchor" id='import'>
<font color = '#006400'>
    
# **2. Data Access, Exploration and Understanding** </font>
</a>

<a class="anchor" id='lib'></a>
<font color = '#008000'>

## **2.1. Ratings** </font>

In [ ]:
def remove_columns(ratings):
    #Remove the columns with no values in it
    for column in ratings.columns:
        if len(ratings[column].value_counts()) == 0:
            ratings.drop(column, axis=1, inplace = True)
    return ratings

def check_ratings(ratings):
    #check if the ratings are falling in between 1 and 5
    indexes = ratings[(ratings['ratings'] < 1) | (ratings['ratings'] > 5)].index
    ratings.drop(indexes,inplace=True)
    return ratings

def check_duplicates(ratings):
    #check if the same user don't evaluate the same movie twice
    for unique_user in ratings['userId'].unique():
        user_ratings = ratings[ratings['userId'] == unique_user]
        indexes = user_ratings[user_ratings['movieId'].duplicated()].index
        ratings.drop(indexes, inplace = True)
    return ratings

def timestamp_modification(ratings):
    #change the data type of the column rating_date and create two columns from the rating_date with the specific year and month --> easier to query
    ratings['rating_date'] = pd.to_datetime(ratings['timestamp'], unit='s')
    ratings['rating_year'] = ratings['rating_date'].dt.year
    ratings['rating_month'] = ratings['rating_date'].dt.month
    return ratings

In [ ]:
ratings

,userId,movieId,rating,timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596
...,...,...,...,...
99995,880,476,3,880175444
99996,716,204,5,879795543
99997,276,1090,1,874795795
99998,13,225,2,882399156


In [ ]:
user_ratings = ratings[ratings['userId'] == 196]
user_ratings

,userId,movieId,rating,timestamp
0,196,242,3,881250949
940,196,393,4,881251863
1133,196,381,4,881251728
1812,196,251,3,881251274
1896,196,655,5,881251793
2374,196,67,5,881252017
6910,196,306,4,881251021
7517,196,238,4,881251820
7842,196,663,5,881251911
10017,196,111,4,881251793


In [ ]:
#checking whether user 196 rated the same movie more than once
user_ratings[user_ratings['movieId'].duplicated()]

,userId,movieId,rating,timestamp


In [ ]:
ratings.head()

,userId,movieId,rating,timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596


In [ ]:
ratings.describe()

,userId,movieId,rating,timestamp
count,100000.00000,100000.000000,100000.000000,1.000000e+05
mean,462.48475,425.530130,3.529860,8.835289e+08
std,266.61442,330.798356,1.125674,5.343856e+06
min,1.00000,1.000000,1.000000,8.747247e+08
25%,254.00000,175.000000,3.000000,8.794487e+08
50%,447.00000,322.000000,4.000000,8.828269e+08
75%,682.00000,631.000000,4.000000,8.882600e+08
max,943.00000,1682.000000,5.000000,8.932866e+08


In [ ]:
ratings.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype
---  ------     --------------   -----
 0   userId     100000 non-null  int64
 1   movieId    100000 non-null  int64
 2   rating     100000 non-null  int64
 3   timestamp  100000 non-null  int64
dtypes: int64(4)
memory usage: 3.1 MB


<a class="anchor" id='lib'></a>
<font color = '#008000'>

## **2.2. Movies** </font>

In [ ]:
movies.head()

,movieId,title,release_date,video_release_date,IMDb_URL,unknown,Action,Adventure,Animation,Children's,...,Fantasy,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,1,Toy Story (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Toy%20Story%2...,0,0,0,1,1,...,0,0,0,0,0,0,0,0,0,0
1,2,GoldenEye (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?GoldenEye%20(...,0,1,1,0,0,...,0,0,0,0,0,0,0,1,0,0
2,3,Four Rooms (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Four%20Rooms%...,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
3,4,Get Shorty (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Get%20Shorty%...,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,5,Copycat (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Copycat%20(1995),0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0


In [ ]:
def remove_columns(movies):
    #Remove the columns with no values in it
    for column in movies.columns:
        if len(movies[column].value_counts()) == 0:
            movies.drop(column, axis=1, inplace = True)
    return movies

In [ ]:
def check_duplicates(movies):
    #check if the same movie appears twice
    indexes = movies[movies.duplicated()].index
    movies.drop(indexes, inplace = True)
    return movies


In [ ]:
def drop_irrelevant_columns(movies):
    #Drop the "video_release_date" column (no value in it)
    movies.drop('video_release_date', axis=1, inplace = True)
    return movies

In [ ]:
movies.columns

Index(['movieId', 'title', 'release_date', 'video_release_date', 'IMDb_URL',
       'unknown', 'Action', 'Adventure', 'Animation', 'Children's', 'Comedy',
       'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror',
       'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War',
       'Western'],
      dtype='object')

In [ ]:
def timestamp_modification(movies):
    #Create 3 date columns from the timestamp column
    import pandas as pd
    movies['release_date'] = pd.to_datetime(movies['release_date'])
    movies['release_date_year']=movies['release_date'].dt.year
    movies['release_date_month']=movies['release_date'].dt.month
    return movies


In [ ]:
def title_modification(df):
    #change the title column to lowercase letters
    df['title']=df['title'].str.lower()
    #remove the year from the title name
    df['title'] = df['title'].str.replace(r'\s*\(\d{4}\)$', '', regex=True)
    return df
movies

,movieId,title,release_date,video_release_date,IMDb_URL,unknown,Action,Adventure,Animation,Children's,...,Fantasy,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,1,Toy Story (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Toy%20Story%2...,0,0,0,1,1,...,0,0,0,0,0,0,0,0,0,0
1,2,GoldenEye (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?GoldenEye%20(...,0,1,1,0,0,...,0,0,0,0,0,0,0,1,0,0
2,3,Four Rooms (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Four%20Rooms%...,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
3,4,Get Shorty (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Get%20Shorty%...,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,5,Copycat (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Copycat%20(1995),0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1677,1678,Mat' i syn (1997),06-Feb-1998,NaN,http://us.imdb.com/M/title-exact?Mat%27+i+syn+...,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1678,1679,B. Monkey (1998),06-Feb-1998,NaN,http://us.imdb.com/M/title-exact?B%2E+Monkey+(...,0,0,0,0,0,...,0,0,0,0,0,1,0,1,0,0
1679,1680,Sliding Doors (1998),01-Jan-1998,NaN,http://us.imdb.com/Title?Sliding+Doors+(1998),0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
1680,1681,You So Crazy (1994),01-Jan-1994,NaN,http://us.imdb.com/M/title-exact?You%20So%20Cr...,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
movies.describe().T

,count,mean,std,min,25%,50%,75%,max
movieId,1682.0,841.500000,485.695893,1.0,421.25,841.5,1261.75,1682.0
video_release_date,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
unknown,1682.0,0.001189,0.034473,0.0,0.00,0.0,0.00,1.0
Action,1682.0,0.149227,0.356418,0.0,0.00,0.0,0.00,1.0
Adventure,1682.0,0.080262,0.271779,0.0,0.00,0.0,0.00,1.0
Animation,1682.0,0.024970,0.156081,0.0,0.00,0.0,0.00,1.0
Children's,1682.0,0.072533,0.259445,0.0,0.00,0.0,0.00,1.0
Comedy,1682.0,0.300238,0.458498,0.0,0.00,0.0,1.00,1.0
Crime,1682.0,0.064804,0.246253,0.0,0.00,0.0,0.00,1.0
Documentary,1682.0,0.029727,0.169882,0.0,0.00,0.0,0.00,1.0


In [ ]:
movies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1682 entries, 0 to 1681
Data columns (total 24 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   movieId             1682 non-null   int64  
 1   title               1682 non-null   object 
 2   release_date        1681 non-null   object 
 3   video_release_date  0 non-null      float64
 4   IMDb_URL            1679 non-null   object 
 5   unknown             1682 non-null   int64  
 6   Action              1682 non-null   int64  
 7   Adventure           1682 non-null   int64  
 8   Animation           1682 non-null   int64  
 9   Children's          1682 non-null   int64  
 10  Comedy              1682 non-null   int64  
 11  Crime               1682 non-null   int64  
 12  Documentary         1682 non-null   int64  
 13  Drama               1682 non-null   int64  
 14  Fantasy             1682 non-null   int64  
 15  Film-Noir           1682 non-null   int64  
 16  Horror

<a class="anchor" id='lib'></a>
<font color = '#008000'>

## **2.3. Users** </font>

In [ ]:
users.head()

,userId,age,gender,occupation,zip_code
0,1,24,M,technician,85711
1,2,53,F,other,94043
2,3,23,M,writer,32067
3,4,24,M,technician,43537
4,5,33,F,other,15213


In [ ]:
users.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 943 entries, 0 to 942
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   userId      943 non-null    int64 
 1   age         943 non-null    int64 
 2   gender      943 non-null    object
 3   occupation  943 non-null    object
 4   zip_code    943 non-null    object
dtypes: int64(2), object(3)
memory usage: 37.0+ KB


In [ ]:
#frequency distribution of all categorical columns
for col in users.select_dtypes(include=['object', 'category']).columns:
    print()
    print(f"--- {col} ---")
    display(users[col].value_counts())


--- gender ---


gender
M    670
F    273
Name: count, dtype: int64


--- occupation ---


occupation
student          196
other            105
educator          95
administrator     79
engineer          67
programmer        66
librarian         51
writer            45
executive         32
scientist         31
artist            28
technician        27
marketing         26
entertainment     18
healthcare        16
retired           14
lawyer            12
salesman          12
none               9
homemaker          7
doctor             7
Name: count, dtype: int64


--- zip_code ---


zip_code
55414    9
55105    6
20009    5
55337    5
10003    5
        ..
55038    1
33319    1
97229    1
78209    1
06405    1
Name: count, Length: 795, dtype: int64

<a class="anchor" id='lib'></a>
<font color = '#008000'>

## **2.4. Genres** </font>

In [ ]:
genres.head()

,genre,genreId
0,unknown,0
1,Action,1
2,Adventure,2
3,Animation,3
4,Children's,4


In [ ]:
genres.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19 entries, 0 to 18
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   genre    19 non-null     object
 1   genreId  19 non-null     int64 
dtypes: int64(1), object(1)
memory usage: 432.0+ bytes


In [ ]:
def remove_columns(genres):
    #Remove the columns with no values in it
    for column in genres.columns:
        if len(genres[column].value_counts()) == 0:
            genres.drop(column, axis=1, inplace = True)
    return genres

def check_duplicates(genres):
    #check if the same user appears twice
    indexes = genres[genres.duplicated()].index
    genres.drop(indexes, inplace = True)
    return genres

In [ ]:
genres.head()

,genre,genreId
0,unknown,0
1,Action,1
2,Adventure,2
3,Animation,3
4,Children's,4


<a class="anchor" id='lib'></a>
<font color = '#008000'>

## **2.5. Occupations** </font>

In [ ]:
occupations.head()

,occupation
0,administrator
1,artist
2,doctor
3,educator
4,engineer


In [ ]:
occupations.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 1 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   occupation  21 non-null     object
dtypes: object(1)
memory usage: 296.0+ bytes


In [ ]:
occupations['occupation'].values

array(['administrator', 'artist', 'doctor', 'educator', 'engineer',
       'entertainment', 'executive', 'healthcare', 'homemaker', 'lawyer',
       'librarian', 'marketing', 'none', 'other', 'programmer', 'retired',
       'salesman', 'scientist', 'student', 'technician', 'writer'],
      dtype=object)

In [ ]:
def remove_columns(occupation):
    #Remove the columns with no values in it
    for column in occupation.columns:
        if len(occupation[column].value_counts()) == 0:
            occupation.drop(column, axis=1, inplace = True)
    return occupation

def check_duplicates(occupation):
    #check if the same user appears twice
    indexes = occupation[occupation.duplicated()].index
    occupation.drop(indexes, inplace = True)
    return occupation

<a class="anchor" id='import'>
<font color = '#006400'>
    
# **3. Convert to Parquet** </font>
</a>

In [ ]:
# Define paths
output_dir = "/Users/cristiana/Downloads/ml-100k/parquet/"

# Make sure the directory exists
import os
os.makedirs(output_dir, exist_ok=True)

# Convert and save
ratings.to_parquet(os.path.join(output_dir, "ratings.parquet"), index=False)
movies.to_parquet(os.path.join(output_dir, "movies.parquet"), index=False)
users.to_parquet(os.path.join(output_dir, "users.parquet"), index=False)
genres.to_parquet(os.path.join(output_dir, "genres.parquet"), index=False)
occupations.to_parquet(os.path.join(output_dir, "occupations.parquet"), index=False)